In [17]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

PROJECT_ROOT = Path("..").resolve()
sys.path.append(str(PROJECT_ROOT))

from src.load_data import load_review
from src.preprocessing import filter_users_items, time_split
from scipy.sparse.linalg import svds
from scipy.sparse import csr_matrix

In [2]:
df = load_review()

In [3]:
filtered_users = filter_users_items(df)

After deduplication:  2,489,395 interactions remaining
After k-core filter:  508,181 interactions remaining


## Train / Validation / Test Split

Time-based split to prevent data leakage — future interactions never inform past predictions.
Only users and items seen in train are kept in val/test (cold-start excluded).

Train covers the earliest 70% of interactions, val the next 15%, and test the final 15%.

We keep time-based split even for SVD (which has no notion of time) because it reflects
real production conditions, prevents metric inflation, and prepares the pipeline for
time-decay and future models. Metrics will be slightly lower than a random split but honest.

In [4]:
train, val, test = time_split(filtered_users)

n_users = filtered_users.reviewerID.nunique()
n_items = filtered_users.asin.nunique()
n_interactions = len(filtered_users)

sparsity = 1 - (n_interactions / (n_users * n_items))

print(f"Users: {n_users}\nItems: {n_items}\nInteractions: {len(filtered_users)}")
print("Sparsity:", sparsity)

Split sizes -> Train: 355760, Val: 32840, Test: 21379
Val coverage:  21.5% of post-t1 interactions kept
Test coverage: 28.0% of post-t2 interactions kept
Users: 59708
Items: 25425
Interactions: 508181
Sparsity: 0.9996652466454411


In [5]:
assert val["reviewerID"].isin(train["reviewerID"]).all()
assert val["asin"].isin(train["asin"]).all()

assert test["reviewerID"].isin(train["reviewerID"]).all()
assert test["asin"].isin(train["asin"]).all()

## Base model strategy

SVD needs no val set (no training loop or early stopping), but we keep the split unchanged.
Training all models on the same 70% ensures fair comparison when moving to future models.
Val will be used for early stopping once we introduce them. Metrics may be slightly
lower than if we trained SVD on 85%, but comparison integrity matters more.

In [19]:
train['user_idx'] = pd.Categorical(train['reviewerID']).codes
train['item_idx'] = pd.Categorical(train['asin']).codes

user_index = dict(enumerate(pd.Categorical(train['reviewerID']).categories))
item_index = dict(enumerate(pd.Categorical(train['asin']).categories))

user_to_idx = {v: k for k, v in user_index.items()}
item_to_idx = {v: k for k, v in item_index.items()}

In [22]:
user_means = train.groupby('reviewerID')['overall'].mean()

train['overall_centered'] = train['overall'] - train['reviewerID'].map(user_means)
sparse = csr_matrix(
    (train['overall_centered'], (train['user_idx'], train['item_idx']))
)

In [23]:
U, sigma, Vt = svds(sparse, k=50)

In [27]:
def predict(user_id, item_id):
    if user_id not in user_to_idx or item_id not in item_to_idx:
        return None
    user_idx = user_to_idx[user_id]
    item_idx = item_to_idx[item_id]
    pred = (U[user_idx] * sigma) @ Vt[:, item_idx]
    return pred + user_means[user_id]

test['pred'] = test.apply(lambda r: predict(r['reviewerID'], r['asin']), axis=1)
test = test.dropna(subset=['pred'])
rmse = np.sqrt(((test['overall'] - test['pred']) ** 2).mean())
print(f"RMSE: {rmse:.4f}")

RMSE: 1.3238


In [26]:
test['pred_mean'] = test['reviewerID'].map(user_means)
rmse_mean = np.sqrt(((test['overall'] - test['pred_mean']) ** 2).mean())
print(f"User mean baseline RMSE: {rmse_mean:.4f}")

User mean baseline RMSE: 1.3239


In [28]:
print(f"Ratings per user: {train.groupby('reviewerID').size().describe()}")

Ratings per user: count    49841.000000
mean         7.137899
std          9.963744
min          1.000000
25%          4.000000
50%          5.000000
75%          8.000000
max        795.000000
dtype: float64


RMSE: 1.3238 vs user-mean baseline 1.3239 — negligible difference.
Median user has only 5 ratings, too sparse for collaborative filtering to find meaningful patterns

After research, LightFM appeared to be a strong fit — it extends matrix factorization
to incorporate item features directly, which aligns well with this dataset's characteristics.